In [1]:
# ─── 【导师特制：向下兼容黄金依赖环境】 ───
# 强制指定 transformers 版本低于 4.45.0（例如 4.44.2），彻底绕过新版的类型死锁
!pip uninstall -y transformers accelerate
!pip install "transformers==4.44.2" accelerate pandas tqdm

print("\n[SUCCESS 🟢] 兼容老模型版的黄金 ASR 环境平铺完毕！")

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: accelerate 1.12.0
Uninstalling accelerate-1.12.0:
  Successfully uninstalled accelerate-1.12.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 94.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 96.1 MB/s eta 0:00:00:00:01
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninsta

In [ ]:
# ======================================================================
# 🚀 Kaggle T4 专属：Wav2Vec2 (300M & 1B) 绝对路径硬核对齐闪电流水线
# ======================================================================
import pandas as pd
import os
import torch
import gc
from transformers import pipeline
from tqdm.notebook import tqdm

# 1. 核心路径配置（完美对齐你给出的 Kaggle 挂载点）
INPUT_CSV = "/kaggle/input/datasets/joyeomm/swissdial/master_metadata.csv"
OUTPUT_300M_CSV = "/kaggle/working/master_metadata_wav2vec_300m_completed.csv"
OUTPUT_1B_CSV = "/kaggle/working/master_metadata_wav2vec_1b_completed.csv"

# 🚨 你的物理音频根目录终极对齐锁
# 因为 CSV 里自带 "data1/"，我们拼上这个前缀后，就会完美变成：
# /kaggle/input/datasets/joyeomm/swissdial/data1/data1/ag/ch_ag_0000.wav
KAGGLE_AUDIO_ROOT = "/kaggle/input/datasets/joyeomm/swissdial/data1"

print("[步骤 1] 正在通过物理硬核对齐，戳穿 Kaggle 路径迷宫...", flush=True)
df_master = pd.read_csv(INPUT_CSV)

# 终极雷达对齐：将表里的相对路径与 Kaggle 深度路径硬核拼装
df_master['real_audio_path'] = df_master['audio_path'].apply(
    lambda x: os.path.join(KAGGLE_AUDIO_ROOT, str(x).strip()).replace("\\", "/")
)

# 哨兵首行通路自检
first_audio = df_master['real_audio_path'].iloc[0]
print(f"[断点哨兵] 系统当前全速锁定的第一条音频绝对物理路径为:\n   👉 '{first_audio}'")

if not os.path.exists(first_audio):
    print(f"\n[🚨 惊天熔断] 依然找不到文件！拼出来的路径是: {first_audio}")
    print("请当场开一个单元格运行 !ls /kaggle/input/datasets/joyeomm/swissdial/data1/ 看看里面到底叫什么！")
    raise FileNotFoundError("物理音频文件路径未对齐，安全熔断。")
else:
    print("[验证通过 🟢] 物理血脉百分之百彻底打通！双轨战车准备起飞。")
    print(f"[任务总量] 总计: {len(df_master)} 条音频。\n" + "-"*70)

device = 0 if torch.cuda.is_available() else -1

# ======================================================================
# 🚄 轨道 A：突击闪击 AndrewMcDowell-300M-German-DE
# ======================================================================
def run_wav2vec_300m():
    print("\n" + "="*70)
    print("[轨道 A] 正在将 Wav2Vec2-300M 德语特制模型载入 T4 显卡...")
    print(f"👉 模型ID: AndrewMcDowell/wav2vec2-xls-r-300m-german-de")
    print("="*70, flush=True)
    
    model_300m_id = "AndrewMcDowell/wav2vec2-xls-r-300m-german-de"
    # pipeline 内部自动接管了多线程多媒体音频解码，绝对不会报 ffmpeg 错误！
    pipe_300m = pipeline("automatic-speech-recognition", model=model_300m_id, device=device)
    print("[SUCCESS ✨] 300M 德语优化版底座就位，开跑！")
    
    write_header = True
    buffer_records = []
    
    # 纯编码器架构，没有自回归延迟，进度条会像心跳一样疯狂刷新闪烁！
    for idx, row in tqdm(df_master.iterrows(), total=len(df_master), desc="300M 轨道推进"):
        audio_file = row['real_audio_path']
        
        if not os.path.exists(audio_file):
            pred_text = "FILE_NOT_FOUND"
        else:
            try:
                result = pipe_300m(audio_file)
                pred_text = result["text"].strip()
            except Exception as e:
                pred_text = f"ERROR_{str(e)[:20]}"
                
        # 严格保持你最原始的 master_metadata CSV 格式，绝不多套娃任何路径
        record = {
            "sentence_id": row['sentence_id'],
            "dialect": row['dialect'],
            "audio_path": row['audio_path'],  # 保持大表原始相对路径形态，不污染大表
            "ref_high_german": row['ref_high_german'],
            "ref_swiss_german": row['ref_swiss_german'],
            "pred_text_wav2vec_300m": pred_text  
        }
        buffer_records.append(record)
        
        # 聚满 64 条追加一次硬盘，保护 Kaggle 文件系统
        if len(buffer_records) >= 64:
            pd.DataFrame(buffer_records).to_csv(OUTPUT_300M_CSV, mode='a', index=False, header=write_header, encoding='utf-8')
            write_header = False
            buffer_records.clear()
            
    if buffer_records:
        pd.DataFrame(buffer_records).to_csv(OUTPUT_300M_CSV, mode='a', index=False, header=write_header, encoding='utf-8')
        
    print(f"\n[🎉 战役A圆满收官] 300M 完美大表已落地保存 -> {OUTPUT_300M_CSV}")

# ======================================================================
# 🚄 轨道 B：接力总攻 JonatasGrosman-1B-German (满血版)
# ======================================================================
def run_wav2vec_1b():
    print("\n" + "="*70)
    print("[轨道 B] 正在将 Wav2Vec2-1B 满血版德语模型载入 T4 显卡...")
    print(f"👉 模型ID: jonatasgrosman/wav2vec2-xls-r-1b-german")
    print("="*70, flush=True)
    
    model_1b_id = "jonatasgrosman/wav2vec2-xls-r-1b-german"
    pipe_1b = pipeline("automatic-speech-recognition", model=model_1b_id, device=device)
    print("[SUCCESS ✨] 1B 满血版编码器顺利空降卡中，开跑！")
    
    write_header = True
    buffer_records = []
    
    for idx, row in tqdm(df_master.iterrows(), total=len(df_master), desc="1B 轨道推进"):
        audio_file = row['real_audio_path']
        
        if not os.path.exists(audio_file):
            pred_text = "FILE_NOT_FOUND"
        else:
            try:
                result = pipe_1b(audio_file)
                pred_text = result["text"].strip()
            except Exception as e:
                pred_text = f"ERROR_{str(e)[:20]}"
                
        record = {
            "sentence_id": row['sentence_id'],
            "dialect": row['dialect'],
            "audio_path": row['audio_path'],  # 保持原始形态
            "ref_high_german": row['ref_high_german'],
            "ref_swiss_german": row['ref_swiss_german'],
            "pred_text_wav2vec_1b": pred_text  
        }
        buffer_records.append(record)
        
        if len(buffer_records) >= 64:
            pd.DataFrame(buffer_records).to_csv(OUTPUT_1B_CSV, mode='a', index=False, header=write_header, encoding='utf-8')
            write_header = False
            buffer_records.clear()
            
    if buffer_records:
        pd.DataFrame(buffer_records).to_csv(OUTPUT_1B_CSV, mode='a', index=False, header=write_header, encoding='utf-8')
        
    print(f"\n[🎉 战役B圆满收官] 1B 全量满血大表已落地保存 -> {OUTPUT_1B_CSV}")

# ======================================================================
# 🏁 自动化指挥中心（串行闪击机制）
# ======================================================================
if __name__ == "__main__":
    # 1. 轰鸣启动第一轨
    run_wav2vec_300m()
    
    # 2. 深度显存垃圾清除：强制切断 300M 模型的缓存残余，腾出 100% 干净显存给 1B，绝对防御 OOM！
    print("\n[显存大清洗] 正在洗净轨道 A 痕迹，全力迎接 1B 战神入驻...")
    torch.cuda.empty_cache()
    gc.collect()
    
    # 3. 轰鸣启动第二轨
    run_wav2vec_1b()
    
    print("\n" + "="*70)
    print("[全面大获全胜 🎉🎉🎉] 恭喜你！Wav2Vec2 双轨道全量长跑圆满大收官！")
    print(f" 👉 300M 特制表保存在: {OUTPUT_300M_CSV}")
    print(f" 👉 1B 满血大表保存在: {OUTPUT_1B_CSV}")
    print("="*70 + "\n")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

2026-05-25 09:33:14.730654: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779701595.130950     654 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779701595.254103     654 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779701596.341291     654 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779701596.341334     654 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779701596.341337     654 computation_placer.cc:177] computation placer alr

[步骤 1] 正在通过物理硬核对齐，戳穿 Kaggle 路径迷宫...
[断点哨兵] 系统当前全速锁定的第一条音频绝对物理路径为:
   👉 '/kaggle/input/datasets/joyeomm/swissdial/data1/data1/ag/ch_ag_0000.wav'
[验证通过 🟢] 物理血脉百分之百彻底打通！双轨战车准备起飞。
[任务总量] 总计: 30921 条音频。
----------------------------------------------------------------------

[轨道 A] 正在将 Wav2Vec2-300M 德语特制模型载入 T4 显卡...
👉 模型ID: AndrewMcDowell/wav2vec2-xls-r-300m-german-de


Some weights of the model checkpoint at AndrewMcDowell/wav2vec2-xls-r-300m-german-de were not used when initializing Wav2Vec2ForCTC: ['wav2vec2.encoder.pos_conv_embed.conv.weight_g', 'wav2vec2.encoder.pos_conv_embed.conv.weight_v']
- This IS expected if you are initializing Wav2Vec2ForCTC from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing Wav2Vec2ForCTC from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at AndrewMcDowell/wav2vec2-xls-r-300m-german-de and are newly initialized: ['wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original1']
You

preprocessor_config.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

[SUCCESS ✨] 300M 德语优化版底座就位，开跑！


300M 轨道推进:   0%|          | 0/30921 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
